# 05 - Conclusiones

Las conclusiones se apoyan en la inspeccion inicial, la limpieza documentada, el EDA y PCA. Se separa evidencia observada de interpretacion y se remarca la logica de trabajo: diagnosticar, limpiar, validar y recien despues interpretar. La idea no es sonar tecnico por sonar tecnico, sino mostrar por que cada decision tiene sentido.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("..").resolve()
raw = pd.read_json(ROOT / "data" / "raw" / "streaming_users_dirty.json")
df = pd.read_csv(ROOT / "data" / "processed" / "streaming_users_processed.csv")
log = pd.read_csv(ROOT / "logs" / "pipeline_log.csv")
log

,Paso,Decision,Evidencia,Filas,Nulos,Retencion (%)
0,0,Carga del dataset original en una copia de tra...,Se preserva data/raw sin modificar.,8160,753,100.00
1,1,Eliminacion de duplicados exactos.,Se eliminaron 126 filas repetidas linea por li...,8034,753,98.46
2,2,Resolucion de user_id repetidos con ranking de...,Se quitaron 34 filas excedentes conservando el...,8000,743,98.04
3,3,Estandarizacion de categorias equivalentes.,"Se unificaron variantes de plan, pais y genero...",8000,743,98.04
4,4,Conversion de valores imposibles a nulos.,"Se marcaron como nulos edades fuera de 13-100,...",8000,1019,98.04
5,5,"Imputacion de nulos con medianas, modas y fech...",La base queda sin nulos y los reemplazos se ap...,8000,0,98.04
6,6,Winsorizacion superior de consumo mensual y ti...,Se aplico cap en percentil 99: watch_time=3628...,8000,0,98.04
7,7,Normalizacion final de tipos y exportacion.,Se exporta el dataset procesado con las mismas...,8000,0,98.04


In [2]:
resumen_calidad = {
    "filas_originales": len(raw),
    "filas_finales": len(df),
    "retencion_final": log.iloc[-1]["Retencion (%)"],
    "nulos_finales": int(df.isna().sum().sum()),
    "duplicados_exactos_finales": int(df.duplicated().sum()),
    "user_id_repetidos_finales": int(df.duplicated("user_id").sum()),
    "mismas_columnas_originales": list(raw.columns) == list(df.columns),
}
resumen_calidad

{'filas_originales': 8160,
 'filas_finales': 8000,
 'retencion_final': np.float64(98.04),
 'nulos_finales': 0,
 'duplicados_exactos_finales': 0,
 'user_id_repetidos_finales': 0,
 'mismas_columnas_originales': True}

In [3]:
resumen_analitico = {
    "consumo_mediano": df["monthly_watch_time_mins"].median(),
    "consumo_promedio": round(df["monthly_watch_time_mins"].mean(), 2),
    "edad_mediana": df["age"].median(),
    "tickets_promedio": round(df["customer_support_tickets"].mean(), 2),
    "planes": df["subscription_plan"].nunique(),
    "paises": df["country"].nunique(),
    "generos": df["favorite_genre"].nunique(),
}
resumen_analitico

{'consumo_mediano': np.float64(772.45),
 'consumo_promedio': np.float64(809.03),
 'edad_mediana': np.float64(33.0),
 'tickets_promedio': np.float64(0.84),
 'planes': 3,
 'paises': 7,
 'generos': 7}

## Hallazgos de calidad

- La base original tenia problemas que podian afectar cualquier analisis: duplicados exactos, `user_id` repetidos, nulos, categorias inconsistentes, fechas invalidas, valores imposibles y extremos. Si eso no se revisaba primero, cualquier conclusion posterior podia quedar apoyada en ruido.
- La limpieza no se hizo de forma automatica sin revisar. Primero se mostro evidencia con codigo y despues se aplico una regla. En otras palabras, no se limpio por costumbre, sino con criterio.
- Para duplicados exactos se uso `raw.duplicated()` antes de eliminar.
- Para `user_id` repetidos se uso un ranking de calidad: fecha valida, consumo plausible, cercania al consumo tipico, login mas reciente y completitud. No se imputaron porque un identificador no es un valor faltante: es una clave que hay que resolver con una sola fila confiable.
- El resultado final conserva 8000 usuarios, 8 columnas originales, 0 nulos y 0 duplicados.

## Interpretacion general

El valor del trabajo no esta solamente en limpiar la base, sino en poder explicar por que se limpio de esa manera. Un analista no deberia borrar datos sin mostrar primero que problema encontro; de lo contrario, la limpieza queda como una decision ciega y no como una decision metodologica.

Por ejemplo, si se quiere limpiar duplicados, antes debe existir codigo que los detecte. En este proyecto eso queda documentado en `02_calidad_y_limpieza.ipynb`, junto con el log de impacto de cada paso. Esa trazabilidad es la que permite defender el trabajo con tranquilidad.

La winsorizacion tampoco se aplico por costumbre. Se uso porque valores extremos de consumo y tickets podian distorsionar medias, graficos, correlaciones y PCA. El corte se justifico con percentil 99 sobre la base ya depurada, y antes de eso los tickets sospechosos se identificaron con una regla robusta basada en IQR. Primero se marco lo sospechoso y despues se acoto lo extremo; asi se conserva informacion real sin dejar que unos pocos valores dominen todo el analisis.

## Hallazgos del analisis

- El consumo mensual es la variable mas directa para describir intensidad de uso.
- La edad aporta contexto, pero no explica por si sola el comportamiento de visualizacion.
- El plan de suscripcion, el genero favorito y los tickets de soporte agregan lectura de perfil: permiten pensar usuarios por consumo, preferencia y friccion operativa.
- PCA ayuda a resumir variables numericas, pero no reemplaza el EDA ni permite afirmar causalidad.
- La imputacion se justifica mejor como un caso MAR que como MCAR, porque los faltantes se condicionan por variables observadas como plan y pais. Dicho de forma simple, no se asumio que faltaban porque si: se busco una explicacion en lo que si estaba observado.


## Limitaciones

- El dataset no incluye churn, fecha de alta, antiguedad, precio pagado, satisfaccion, dispositivo ni historial detallado de sesiones.
- Las conclusiones son descriptivas: muestran patrones observados, no causas definitivas.
- La imputacion y la winsorizacion son decisiones justificadas, pero siguen siendo decisiones analiticas que deben declararse; en este proyecto se apoyan en medianas, modas, IQR y percentiles, con un cap final por percentil 99 en variables sesgadas. No se trata de eliminar problemas, sino de tratarlos sin exagerar su impacto.
- La fecha de ultimo login permite una lectura parcial de actividad, pero no alcanza para medir retencion real.


## Proximos pasos

- Incorporar variables comerciales como precio, promociones, fecha de alta y cancelaciones.
- Agregar metricas temporales: cantidad de sesiones, dias activos y evolucion mensual del consumo.
- Analizar retencion o riesgo de baja si se incorpora una variable objetivo.
- Validar las reglas de limpieza con una mirada de negocio antes de usar el proceso en produccion.
- Automatizar controles de calidad para detectar duplicados, nulos y rangos imposibles en futuras cargas.

## Conclusion final

El proyecto deja una base ordenada, trazable y lista para analisis. La evidencia muestra que el comportamiento de usuarios de streaming no puede resumirse con una sola variable: consumo, soporte, plan, edad y genero favorito aportan piezas distintas del perfil.

La conclusion mas importante es metodologica: antes de interpretar, hay que preparar bien los datos. Sin esa etapa, el analisis puede sonar convincente, pero apoyarse en errores de carga, duplicados o extremos poco realistas. Si tuviera que decirlo en una frase mas humana, seria esta: primero dejo el dato en orden, despues me animo a contarlo.